In [1]:
import pandas as pd
import time
import os
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# Setup Headless Chrome
options = Options()
options.add_argument('--headless')
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

print("Installing ChromeDriver...")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
print("Driver setup complete.")

Installing ChromeDriver...
Driver setup complete.


In [2]:
try:
    url = "https://www.leechiu.com/property-search?propertyType=all&location=all&search=&listingType=For%20Sale"
    print(f"Navigating to {url}...")
    driver.get(url)
    
    time.sleep(5)
    
    print("Scrolling to load all properties...")
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            time.sleep(2)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
        last_height = new_height
        
    print("Finished scrolling. Extracting data...")
    
    listings_data = []
    batch_filename = "leechiu_properties_partial.csv"
    
    all_elements = driver.find_elements(By.XPATH, "//div[contains(@class, 'flex') and contains(@class, 'flex-wrap')]//div[contains(@class, 'mt-8')]")
    print(f"Found {len(all_elements)} potential cards.")

    print("Processing cards:", end=" ")
    for i, card in enumerate(all_elements):
        if i < 10:
             print(f"{i+1}...", end=" ", flush=True)
        
        try:
            text_content = card.text.split('\n')
            
            title = ""
            price = ""
            location = ""
            description = ""
            link = ""
            
            try:
                link_el = card.find_element(By.TAG_NAME, "a")
                link = link_el.get_attribute("href")
            except:
                pass

            # Improved Logic:
            lines = [l.strip() for l in text_content if l.strip()]
            
            for idx, line in enumerate(lines):
                line_upper = line.upper()
                
                # Price
                if "ASKING PRICE" in line_upper or "LEASE RATE" in line_upper:
                    if idx + 1 < len(lines):
                         if ":" in line:
                             price = line.split(":", 1)[1].strip()
                         else:
                             price = lines[idx+1]
                
                # Location
                if "ADDRESS" in line_upper:
                    if ":" in line:
                         location = line.split(":", 1)[1].strip()
                    elif idx + 1 < len(lines):
                         location = lines[idx+1]

                # Description
                if "DESCRIPTION" in line_upper:
                     if ":" in line:
                         description = line.split(":", 1)[1].strip()
                     elif idx + 1 < len(lines):
                         description = lines[idx+1]

            # 3. Title Logic Corrected
            # Only ignore strictly status tags or explicit field labels
            status_tags = ["FOR SALE", "FOR LEASE"]
            field_labels = ["ASKING PRICE", "LEASE RATE", "ADDRESS", "DESCRIPTION", "PHP"]
            
            for line in lines:
                line_u = line.upper()
                
                # Skip if it is EXACTLY a status tag
                if line_u in status_tags:
                    continue
                
                # Skip if it contains a field label
                is_label = False
                for label in field_labels:
                    if label in line_u:
                        is_label = True
                        break
                if is_label:
                    continue
                
                # Skip purely numeric/currency lines
                if line.replace(',','').replace('.','').strip().isdigit():
                    continue
                    
                # If we passed filters, this is likely the title
                title = line
                break
            
            listings_data.append({
                "title": title,
                "price": price,
                "location": location,
                "description": description,
                "link": link
            })
            
            if (i + 1) % 10 == 0:
                print(f"\n[Status] Scraped {i + 1} cards. ", end="")
                print("Processing cards:", end=" ")
                
            if (i + 1) % 50 == 0:
                temp_df = pd.DataFrame(listings_data)
                temp_df.to_csv(batch_filename, index=False)
                print(f"(Checkpoint saved)", end="")
            
        except Exception as e:
            print(f"\nError on card {i}: {e}")
            continue
            
    print("\nExtraction complete.")
    df = pd.DataFrame(listings_data)
    print(f"Extracted {len(df)} listings.")
    display(df.head())
    
    final_filename = "leechiu_properties_final.csv"
    df.to_csv(final_filename, index=False)
    print(f"Final data saved to {final_filename}")
    
    if os.path.exists(batch_filename):
        os.remove(batch_filename)
    
except Exception as e:
    print(f"An error occurred: {e}")
    if 'listings_data' in locals() and len(listings_data) > 0:
        pd.DataFrame(listings_data).to_csv("leechiu_properties_crash_recovery.csv", index=False)
finally:
    driver.quit()
    print("Driver closed.")

Navigating to https://www.leechiu.com/property-search?propertyType=all&location=all&search=&listingType=For%20Sale...
Scrolling to load all properties...
Finished scrolling. Extracting data...
Found 390 potential cards.
Processing cards: 1... 2... 3... 4... 5... 6... 7... 8... 9... 10... 
[Status] Scraped 10 cards. Processing cards: 
[Status] Scraped 20 cards. Processing cards: 
[Status] Scraped 30 cards. Processing cards: 
[Status] Scraped 40 cards. Processing cards: 
[Status] Scraped 50 cards. Processing cards: (Checkpoint saved)
[Status] Scraped 60 cards. Processing cards: 
[Status] Scraped 70 cards. Processing cards: 
[Status] Scraped 80 cards. Processing cards: 
[Status] Scraped 90 cards. Processing cards: 
[Status] Scraped 100 cards. Processing cards: (Checkpoint saved)
[Status] Scraped 110 cards. Processing cards: 
[Status] Scraped 120 cards. Processing cards: 
[Status] Scraped 130 cards. Processing cards: 
[Status] Scraped 140 cards. Processing cards: 
[Status] Scraped 150 card

,title,price,location,description,link
0,"1BR CONDOMINIUM FOR SALE IN SEA RESIDENCES, PASAY","PHP 5,415,000","Pearl Drive Cor Sunrise Drive, Pasay, 1300 Met...",1BR with balcony,
1,"1BR CONDOMINIUM FOR SALE IN JAZZ RESIDENCES, M...","PHP 4,825,800","Jupiter St., cor. N. Garcia St., Bel-Air Villa...","43rd Floor, 1BR with balcony",
2,LOT FOR SALE IN BOGO CITY CEBU,"PHP 5,000,000,000","Bogo City Hall Road, Bogo, Cebu","Bogo City, Cebu",
3,"2BR CONDOMINIUM FOR SALE IN DISCOVERY PRIMEA, ...","PHP 220,000,000",Makati,A furnished 2 bedroom condominium unit with 2 ...,
4,LIIP - RRI,PHP NAN,"LIIP, Mamplasan, Binan, Laguna",LIIP lot for lease,


Final data saved to leechiu_properties_final.csv
Driver closed.
